# Ridge Residual Analysis — Read-Only Audit

Verifies RRA v1.0 methodology after counterintuitive ACF increase. **No experiments modified. No GRU training.**

## 1. Setup

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO = Path('..').resolve()
sys.path.insert(0, str(REPO))

RRA_EXP = REPO / 'experiments/ridge_residual_analysis_2026-07-26_161500'
AUDIT_EXP = REPO / 'experiments/ridge_residual_analysis_audit_2026-07-26_162500'

with (AUDIT_EXP / 'reports' / 'audit_report.json').open() as fh:
    report = json.load(fh)
report.keys()

## 2. Task 1 — Residual definition & scaling

In [ ]:
scale = pd.read_csv(AUDIT_EXP / 'verification' / 'task1_scaling_mismatch.csv')
task1 = pd.read_csv(AUDIT_EXP / 'verification' / 'task1_residual_definition.csv')
print('Identity holds (original RRA):', task1['original_identity_holds'].all())
print('Sign: prophet - ridge (confirmed)')
print('Scaling bug: raw prophet minus z-scored ridge')
scale[['container_id','prophet_raw_std','ridge_pred_z_std','mean_abs_diff_raw_vs_z_pred']].head()

## 3. Task 2 — Alignment (Day-1 block)

In [ ]:
with (AUDIT_EXP / 'verification' / 'task2_day1_alignment_sample.json').open() as fh:
    align = json.load(fh)
align

## 4. Task 3 — Data splits / leakage

In [ ]:
with (AUDIT_EXP / 'verification' / 'task3_data_splits.json').open() as fh:
    splits = json.load(fh)
splits

## 5. Task 4 — ACF replication (RRA vs corrected)

In [ ]:
rra = report['rra_reported']
corr = report['corrected_summary']
pd.DataFrame([
    {'metric': 'mean |ACF| 1-10', 'prophet': rra['prophet_mean_acf_1_10'],
     'rra_remaining': rra['rra_remaining_mean_acf_1_10'],
     'corrected_remaining': corr['corrected_remaining_mean_acf_1_10']},
    {'metric': 'Ljung reject lag20', 'prophet': rra['rra_prophet_lb_reject_20'],
     'rra_remaining': rra['rra_remaining_lb_reject_20'],
     'corrected_remaining': corr['corrected_remaining_lb_reject_20']},
])

## 6. Task 5 — Why ACF appeared to increase

In [ ]:
report['task5']

## 7. Task 6 — Linear models on corrected remaining

In [ ]:
ar = pd.read_csv(AUDIT_EXP / 'tables' / 'linear_model_ar_summary.csv')
ar

## 8. Task 7 — Decision

In [ ]:
report['decision']

## 9. Interpretation

In [ ]:
print('Scaling bug confirmed:', report['decision']['scaling_bug_in_rra_v1'])
print('Corrected ACF reduction:', report['corrected_summary']['corrected_acf_reduction'])
print('Ridge→GRU justified:', report['decision']['q3_ridge_gru_justified_vs_more_linear_modelling'])